## Importing libraries

In [ ]:
import qkit #filepath = r'C:\Users\weideslab-admin\AppData\Roaming\Python\Python313\site-packages\qkit\drivers
qkit.cfg['load_visa'] = True # Enables VISA support, which is needed to communicate with lab instruments
qkit.cfg['datafolder_structure'] = 2 # Tells Qkit what folder hierarchy to use when saving experimental data
# New Qkit does not create folder. Must be done before run! (think a txt doc is created which causes this)
qkit.cfg['datadir'] = r'd:\notebooks\Amy' # replace this with your directory
qkit.cfg['run_id'] = 'results_h5' # will look for directory results_h5
qkit.cfg['user'] = 'Amy' # will look for directory [user]
# i.e., should have [directory of notebook folder]/[results_h5]/[user] 
qkit.start()

from qkit.measure.spectroscopy import spectroscopy
from qkit.storage.store import Data
import qkit.measure.samples_class as sc
from qkit.analysis.circle_fit.circle_fit_2019 import circuit

# Uses SciPy for signal processing and curve fitting
from scipy import signal as sg
from scipy.optimize import curve_fit
import qkit.analysis.qfit as qfit

from importlib import reload
import qkit.gui.notebook.Progress_Bar as Pb
import qkit.measure.spectroscopy.spectroscopy as spectroscopy
from importlib import reload
import time

# import skrf as rf
import pyvisa as visa
from tqdm import tqdm
import requests

import mcculw # DAQ 
# import mcculw.ul
from attenuatorControl import ldacontroller

import h5py
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.animation import FuncAnimation
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap

import os
import re
import sys
import io

## Instrument setup

In [ ]:
rm = visa.ResourceManager() # Initialize the VISA library
resources = rm.list_resources() # List available VISA resources

for resource in resources: # Print the list of resources
    print(resource)

In [ ]:
caen = qkit.instruments.create('caen', 'Caen_FAST_PS', address='10.22.197.101')
vna = qkit.instruments.create('vna', 'RS_VNA_ZNLXXX', address='GPIB0::20::INSTR')

In [ ]:
smpl = sc.Sample() # creates sample object (i.e., container for metadata)
s = spectroscopy.spectrum(vna = vna, exp_name = '', sample = smpl)

## Phase Shifter Setup

In [ ]:
from phaseControl import lps_phasecontrolmk2

In [ ]:
vnx, Devices = lps_phasecontrolmk2.retrieveDeviceProperties()

device_handles = list(Devices)
device = device_handles[0]  # first device
print(device_handles)

# Now you can set frequency/phase
vnx.fnLPS_SetWorkingFrequency(device, 5000) # Hz
vnx.fnLPS_SetPhaseAngle(device, 0) # degrees

## VNA Instructions

In [ ]:
# configure vna to see resonator first mode
vna.set_bandwidth(500) # set the bandwidth of the IF filter (measurement bandwidth). Values between 1 Hz and 500 kHz can be set.
vna.set_power(0) # this configures the optional receiver step attenuators
vna.set_startfreq(6.2e9) # set the start frequency of the sweep Hz
vna.set_stopfreq(6.6e9) # set the stop frequency of the sweep Hz
vna.set_nop(200) # set the number of points in the sweep. Values between 1 and 5001 can be set.
vna.set_averages(1) # set the number of averages. Values between 1 and 1 to 1000 can be set.
vna.set_sweep_mode('cont') # set the sweep mode to continuous

In [ ]:
vna.set_sweep_mode('hold') # stops VNA sweep

## Attenuator

In [ ]:
ldacontroller.CLI_Vaunix_Attn().main(['-c', '1', '-a', '0']) # This sets the attenuation. The last parameter is attenuation in dB.

## 1D Measurement

In [ ]:
def get_or_measure_1D(s, filename=None):
    """
    Load an existing 1D measurement HDF5 file or run the measurement if file doesn't exist.
    Returns:
        hdf: Data object
        amp_dB: amplitude in dB
        phase_deg: phase in degrees
        file_path: HDF5 filepath
    """
    if filename is not None and os.path.exists(filename):
        # File exists, just load
        print(f"Loading existing measurement from: {filename}")
        hdf = Data(filename)
        file_path = filename
    else:
        # File does not exist, run measurement
        print("File not found. Running 1D measurement...")
        
        # Capture printed output to get HDF5 path
        old_stdout = sys.stdout
        sys.stdout = mystdout = io.StringIO()
        
        s.measure_1D()
        
        sys.stdout = old_stdout
        output = mystdout.getvalue()
        
        # Extract HDF5 path from printed output
        match = re.search(r"(?:[a-zA-Z]:\\[^\s]+\.h5)", output)
        if not match:
            raise ValueError("Could not detect output HDF5 file path.")
        file_path = match.group(0)
        print(f"Measurement saved to: {file_path}")
        
        hdf = Data(file_path)
    # Extract amplitude and phase
    amp = np.transpose(hdf.data.amplitude[:])
    phase = np.transpose(hdf.data.phase[:])
    
    amp_dB = 20 * np.log10(amp)
    phase_deg = phase * 360 / (2*np.pi)
    
    return hdf, amp_dB, phase_deg, file_path

In [ ]:
hdf, amp_dB, phase_deg, file = get_or_measure_1D(s)

# OR can define filename to load specific file
# hdf, amp_dB, phase_deg, file = get_or_measure_1D(
#     s, 
#     filename=r'd:\notebooks\Amy\RESULTS_H5\Amy\T99WZG_VNA_tracedata\T99WZG_VNA_tracedata.h5'
# )

## 2D Measurement

Similar to 1D

In [ ]:
def x_func(x,s):
    if s == 'phase':
        lps_phasecontrolmk2.vnx.fnLPS_SetPhaseAngle(Devices[0], int(x))
    elif s == 'current':
        caen.ramp_current(x, 1e-1)
    # optionally add a small delay if the phase shifter needs settling
    # time.sleep(0.05)
    return 

In [ ]:
def get_or_measure_2D(s, filename=None):
    """
    Load an existing 2D measurement HDF5 file or run the measurement if file doesn't exist.
    Returns:
        hdf: Data object
        amp_dB: amplitude in dB
        phase_deg: phase in degrees
        file_path: HDF5 filepath
    """
    if filename is not None and os.path.exists(filename):
        # File exists, just load
        print(f"Loading existing measurement from: {filename}")
        hdf = Data(filename)
        file_path = filename
    else:
        # File does not exist, run measurement
        print("File not found. Running 2D measurement...")
        
        # Capture printed output to get HDF5 path
        old_stdout = sys.stdout
        sys.stdout = mystdout = io.StringIO()

        s.set_x_parameters(x_vec = np.arange(0,361,10), # change this to go from 0 to 360 in X steeps
                  x_coordname = 'angle',
                  x_set_obj = x_func,
                  x_unit = 'degrees')
        
        s.measure_2D()
        
        sys.stdout = old_stdout
        output = mystdout.getvalue()
        
        # Extract HDF5 path from printed output
        match = re.search(r"(?:[a-zA-Z]:\\[^\s]+\.h5)", output)
        if not match:
            raise ValueError("Could not detect output HDF5 file path.")
        file_path = match.group(0)
        print(f"Measurement saved to: {file_path}")
        
        hdf = Data(file_path)
    # Extract amplitude and phase
    amp = np.transpose(hdf.data.amplitude[:])
    phase = np.transpose(hdf.data.phase[:])
    
    amp_dB = 20 * np.log10(amp)
    phase_deg = phase * 360 / (2*np.pi)
    
    return hdf, amp_dB, phase_deg, file_path